In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [19]:
app_train = pd.read_csv('data/application_train.csv')
app_test = pd.read_csv('data/application_test.csv')
bureau = pd.read_csv('data/bureau.csv')
bureau_balance = pd.read_csv('data/bureau_balance.csv')
credit_card_balance = pd.read_csv('data/credit_card_balance.csv')
#HC_columns_description = pd.read_csv('data/HomeCredit_columns_description.csv', encoding='cp1252')
installments_payments = pd.read_csv('data/installments_payments.csv')
pos_cash = pd.read_csv('data/POS_CASH_balance.csv')
previous_application = pd.read_csv('data/previous_application.csv')


In [20]:
print(f"Размеры данных:")
print(f"Train: {app_train.shape}, Test: {app_test.shape}")

Размеры данных:
Train: (307511, 122), Test: (48744, 121)


In [21]:
for df in [app_train, app_test]:
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

In [22]:
for df in [app_train, app_test]:
    # Кредит относительно дохода
    df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    # Аннуитет относительно дохода
    df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    # Возраст в годах
    df['DAYS_BIRTH_YEARS'] = abs(df['DAYS_BIRTH'] / 365.25)
    # Стаж работы в годах
    df['DAYS_EMPLOYED_YEARS'] = abs(df['DAYS_EMPLOYED'] / 365.25)
    # Отношение кредита к аннуитету
    df['CREDIT_ANNUITY_RATIO'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'] + 1)
    # Отношение дохода к количеству членов семьи
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)

/tmp/ipykernel_63371/3380182031.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
/tmp/ipykernel_63371/3380182031.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
/tmp/ipykernel_63371/3380182031.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all co

In [23]:
# ============================================================================
# 3. АГРЕГАЦИЯ BUREAU (Кредитная история до Home Credit)
# ============================================================================
print("\nАгрегация bureau...")

# Агрегация bureau_balance - исправленная версия
bureau_balance_agg = bureau_balance.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': ['min', 'max', 'size'],
    'STATUS': [
        ('status_2_count', lambda x: (x == '2').sum()),  # Просрочки 30-59 дней
        ('status_3_count', lambda x: (x == '3').sum()),  # Просрочки 60-89 дней
        ('status_4_count', lambda x: (x == '4').sum()),  # Просрочки 90-119 дней
        ('status_5_count', lambda x: (x == '5').sum())   # Просрочки 120+ дней
    ]
})

# Упрощаем имена колонок
bureau_balance_agg.columns = [
    'BUREAU_BAL_MONTHS_MIN',
    'BUREAU_BAL_MONTHS_MAX', 
    'BUREAU_BAL_MONTHS_COUNT',
    'BUREAU_BAL_STATUS_2',
    'BUREAU_BAL_STATUS_3',
    'BUREAU_BAL_STATUS_4',
    'BUREAU_BAL_STATUS_5'
]
bureau_balance_agg.reset_index(inplace=True)

# Мерджим с bureau
bureau = bureau.merge(bureau_balance_agg, on='SK_ID_BUREAU', how='left')

# Основная агрегация bureau - исправленная версия
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'DAYS_CREDIT': ['min', 'max', 'mean', 'std'],
    'CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_MAX_OVERDUE': ['max', 'mean', 'sum'],
    'CNT_CREDIT_PROLONG': ['sum', 'max'],
    'AMT_CREDIT_SUM': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_OVERDUE': ['sum', 'mean'],
    'AMT_CREDIT_SUM_LIMIT': ['sum', 'mean'],
    'AMT_ANNUITY': ['max', 'mean'],
    'DAYS_CREDIT_ENDDATE': ['min', 'max'],
    'DAYS_ENDDATE_FACT': ['min', 'max'],
    'BUREAU_BAL_STATUS_2': 'sum',
    'BUREAU_BAL_STATUS_3': 'sum',
    'BUREAU_BAL_STATUS_4': 'sum',
    'BUREAU_BAL_STATUS_5': 'sum'
})

bureau_agg.columns = pd.Index([f'BUREAU_{e[0]}_{e[1].upper()}' 
                                for e in bureau_agg.columns])
bureau_agg.reset_index(inplace=True)

# Создаем дополнительные признаки
bureau_agg['BUREAU_LOANS_COUNT'] = bureau.groupby('SK_ID_CURR').size().reset_index(name='size')['size']
bureau_agg['BUREAU_ACTIVE_LOANS'] = bureau[bureau['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR').size().reset_index(name='size')['size']
bureau_agg['BUREAU_CLOSED_LOANS'] = bureau[bureau['CREDIT_ACTIVE'] == 'Closed'].groupby('SK_ID_CURR').size().reset_index(name='size')['size']
bureau_agg['BUREAU_BAD_DEBT_RATIO'] = bureau_agg['BUREAU_CLOSED_LOANS'] / (bureau_agg['BUREAU_LOANS_COUNT'] + 1)

# Мерджим с главной таблицей
app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(bureau_agg, on='SK_ID_CURR', how='left')

# Флаг наличия кредитной истории
for df in [app_train, app_test]:
    df['HAS_BUREAU'] = df['BUREAU_LOANS_COUNT'].notna().astype(int)


Агрегация bureau...


/tmp/ipykernel_63371/1677685128.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_BUREAU'] = df['BUREAU_LOANS_COUNT'].notna().astype(int)
/tmp/ipykernel_63371/1677685128.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_BUREAU'] = df['BUREAU_LOANS_COUNT'].notna().astype(int)


In [24]:
previous_agg = previous_application.groupby('SK_ID_CURR').agg({
    'AMT_ANNUITY': ['sum', 'mean', 'max', 'std'],
    'AMT_APPLICATION': ['sum', 'mean', 'max'],
    'AMT_CREDIT': ['sum', 'mean', 'max'],
    'AMT_DOWN_PAYMENT': ['sum', 'mean', 'max'],
    'AMT_GOODS_PRICE': ['sum', 'mean', 'max'],
    'HOUR_APPR_PROCESS_START': ['min', 'max', 'mean'],
    'DAYS_DECISION': ['min', 'max', 'mean'],
    'CNT_PAYMENT': ['sum', 'mean', 'max']
})

previous_agg.columns = pd.Index([f'PREV_{e[0]}_{e[1].upper()}' 
                                  for e in previous_agg.columns])
previous_agg.reset_index(inplace=True)

# Дополнительные признаки
previous_agg['PREV_APPLICATIONS_COUNT'] = previous_application.groupby('SK_ID_CURR').size().reset_index(name='size')['size']
previous_agg['PREV_APPROVED_COUNT'] = previous_application[previous_application['NAME_CONTRACT_STATUS'] == 'Approved'].groupby('SK_ID_CURR').size().reset_index(name='size')['size']
previous_agg['PREV_REFUSED_COUNT'] = previous_application[previous_application['NAME_CONTRACT_STATUS'] == 'Refused'].groupby('SK_ID_CURR').size().reset_index(name='size')['size']
previous_agg['PREV_APPROVAL_RATE'] = previous_agg['PREV_APPROVED_COUNT'] / (previous_agg['PREV_APPLICATIONS_COUNT'] + 1)

# Мерджим с главной таблицей
app_train = app_train.merge(previous_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(previous_agg, on='SK_ID_CURR', how='left')

# Флаг наличия предыдущих заявок
for df in [app_train, app_test]:
    df['HAS_PREVIOUS_APPLICATION'] = df['PREV_APPLICATIONS_COUNT'].notna().astype(int)

/tmp/ipykernel_63371/3244478950.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_PREVIOUS_APPLICATION'] = df['PREV_APPLICATIONS_COUNT'].notna().astype(int)
/tmp/ipykernel_63371/3244478950.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_PREVIOUS_APPLICATION'] = df['PREV_APPLICATIONS_COUNT'].notna().astype(int)


In [26]:
pos_cash_agg = pos_cash.groupby('SK_ID_CURR').agg({
    'MONTHS_BALANCE': ['min', 'max', 'size'],
    'SK_DPD': ['sum', 'mean', 'max'],      # Исправлено: SK_DPD вместо SKDPD
    'SK_DPD_DEF': ['sum', 'mean', 'max'],  # Исправлено: SK_DPD_DEF вместо SKDPD_DEF
    'CNT_INSTALMENT': ['sum', 'mean', 'max'],
    'CNT_INSTALMENT_FUTURE': ['sum', 'mean', 'max']
})

pos_cash_agg.columns = pd.Index([f'POS_{e[0]}_{e[1].upper()}' 
                                  for e in pos_cash_agg.columns])
pos_cash_agg.reset_index(inplace=True)

pos_cash_agg['POS_CONTRACTS_COUNT'] = pos_cash.groupby('SK_ID_CURR')['SK_ID_PREV'].nunique().reset_index(name='size')['size']

app_train = app_train.merge(pos_cash_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(pos_cash_agg, on='SK_ID_CURR', how='left')

for df in [app_train, app_test]:
    df['HAS_POS'] = df['POS_CONTRACTS_COUNT'].notna().astype(int)

/tmp/ipykernel_63371/2065133971.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_POS'] = df['POS_CONTRACTS_COUNT'].notna().astype(int)
/tmp/ipykernel_63371/2065133971.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_POS'] = df['POS_CONTRACTS_COUNT'].notna().astype(int)


In [28]:
installments = installments_payments

In [30]:
# ============================================================================
# 6. АГРЕГАЦИЯ INSTALLMENTS (Платежи по рассрочке)
# ============================================================================
print("\nАгрегация installments...")

# 1. Создаем признаки отклонения от графика (ИСПРАВЛЕНО: добавлено создание DBD)
installments['PAYMENT_DIFF'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']
installments['DPD'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']      # > 0 означает просрочку
installments['DBD'] = installments['DAYS_INSTALMENT'] - installments['DAYS_ENTRY_PAYMENT']      # > 0 означает досрочный платеж

# 2. Агрегация
installments_agg = installments.groupby('SK_ID_CURR').agg({
    'NUM_INSTALMENT_VERSION': ['nunique'],
    'DPD': ['max', 'mean', 'sum'],
    'DBD': ['max', 'mean', 'sum'],          # Теперь эта колонка существует
    'PAYMENT_DIFF': ['max', 'mean', 'sum'],
    'AMT_INSTALMENT': ['sum', 'mean', 'max'],
    'AMT_PAYMENT': ['sum', 'mean', 'max'],
    'DAYS_INSTALMENT': ['min', 'max', 'mean'],
    'DAYS_ENTRY_PAYMENT': ['min', 'max', 'mean']
})

# 3. Переименование колонок
installments_agg.columns = pd.Index([f'INSTAL_{e[0]}_{e[1].upper()}' 
                                      for e in installments_agg.columns])
installments_agg.reset_index(inplace=True)

# 4. Дополнительный признак: общее количество платежей
installments_agg['INSTAL_PAYMENTS_COUNT'] = installments.groupby('SK_ID_CURR').size().reset_index(name='size')['size']

# 5. Мердж с главными таблицами
app_train = app_train.merge(installments_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(installments_agg, on='SK_ID_CURR', how='left')

# 6. Флаг наличия истории платежей
for df in [app_train, app_test]:
    df['HAS_INSTALMENTS'] = df['INSTAL_PAYMENTS_COUNT'].notna().astype(int)


Агрегация installments...


/tmp/ipykernel_63371/3370525255.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_INSTALMENTS'] = df['INSTAL_PAYMENTS_COUNT'].notna().astype(int)
/tmp/ipykernel_63371/3370525255.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_INSTALMENTS'] = df['INSTAL_PAYMENTS_COUNT'].notna().astype(int)


In [32]:
credit_card = credit_card_balance

In [33]:
# ============================================================================
# 7. АГРЕГАЦИЯ CREDIT_CARD (Кредитные карты)
# ============================================================================
print("\nАгрегация credit_card...")

credit_card_agg = credit_card.groupby('SK_ID_CURR').agg({
    'MONTHS_BALANCE': ['min', 'max', 'size'],
    'AMT_BALANCE': ['max', 'mean'],
    'AMT_CREDIT_LIMIT_ACTUAL': ['max', 'mean'],
    'AMT_DRAWINGS_ATM_CURRENT': ['sum', 'max', 'mean'],
    'AMT_DRAWINGS_CURRENT': ['sum', 'max', 'mean'],
    'AMT_INST_MIN_REGULARITY': ['max', 'mean'],
    'AMT_PAYMENT_CURRENT': ['sum', 'max', 'mean'],
    'AMT_PAYMENT_TOTAL_CURRENT': ['sum', 'max', 'mean'],
    'AMT_RECEIVABLE_PRINCIPAL': ['max', 'mean'],
    'CNT_DRAWINGS_CURRENT': ['sum', 'max', 'mean'],
    'CNT_INSTALMENT_MATURE_CUM': ['max', 'mean'],
    'SK_DPD': ['max', 'mean', 'sum'],      # Правильное название
    'SK_DPD_DEF': ['max', 'mean', 'sum']   # Правильное название
})

credit_card_agg.columns = pd.Index([f'CC_{e[0]}_{e[1].upper()}' 
                                     for e in credit_card_agg.columns])
credit_card_agg.reset_index(inplace=True)

credit_card_agg['CC_CONTRACTS_COUNT'] = credit_card.groupby('SK_ID_CURR')['SK_ID_PREV'].nunique().reset_index(name='size')['size']

app_train = app_train.merge(credit_card_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(credit_card_agg, on='SK_ID_CURR', how='left')

for df in [app_train, app_test]:
    df['HAS_CREDIT_CARD'] = df['CC_CONTRACTS_COUNT'].notna().astype(int)


Агрегация credit_card...


/tmp/ipykernel_63371/2011446667.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_CREDIT_CARD'] = df['CC_CONTRACTS_COUNT'].notna().astype(int)
/tmp/ipykernel_63371/2011446667.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['HAS_CREDIT_CARD'] = df['CC_CONTRACTS_COUNT'].notna().astype(int)


In [34]:
print("\nФинальная обработка...")

# Заполняем пропуски
for df in [app_train, app_test]:
    # Числовые признаки - заполняем -1 (для деревьев это норм)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(-1)
    
    # Категориальные признаки - заполняем 'Missing'
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna('Missing')



Финальная обработка...


/tmp/ipykernel_63371/1773953092.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns
/tmp/ipykernel_63371/1773953092.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes

In [35]:
# Label Encoding для категориальных признаков
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
cat_cols = app_train.select_dtypes(include=['object']).columns

for col in cat_cols:
    if col == 'TARGET':
        continue
    
    le = LabelEncoder()
    # Объединяем train и test для корректного кодирования
    combined = pd.concat([app_train[col], app_test[col]], axis=0)
    le.fit(combined.astype(str))
    
    app_train[col] = le.transform(app_train[col].astype(str))
    app_test[col] = le.transform(app_test[col].astype(str))
    label_encoders[col] = le

print(f"Финальные размеры:")
print(f"Train: {app_train.shape}")
print(f"Test: {app_test.shape}")

/tmp/ipykernel_63371/1864954547.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = app_train.select_dtypes(include=['object']).columns


Финальные размеры:
Train: (307511, 271)
Test: (48744, 270)


In [36]:
app_train

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CC_CNT_INSTALMENT_MATURE_CUM_MAX,CC_CNT_INSTALMENT_MATURE_CUM_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_MEAN,CC_SK_DPD_SUM,CC_SK_DPD_DEF_MAX,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_SUM,CC_CONTRACTS_COUNT,HAS_CREDIT_CARD
0,100002,1,0,1,0,1,0,202500.0,406597.5,24700.5,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
1,100003,0,0,0,0,0,0,270000.0,1293502.5,35698.5,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
2,100004,0,1,1,1,1,0,67500.0,135000.0,6750.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
3,100006,0,0,0,0,1,0,135000.0,312682.5,29686.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
4,100007,0,0,1,0,1,0,121500.0,513000.0,21865.5,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,0,1,0,0,0,157500.0,254700.0,27558.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
307507,456252,0,0,0,0,1,0,72000.0,269550.0,12001.5,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
307508,456253,0,0,0,0,1,0,153000.0,677664.0,29979.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0
307509,456254,1,0,0,0,1,0,171000.0,370107.0,20205.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0


In [37]:
X_train = app_train.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y_train = app_train['TARGET']

In [38]:
cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']


In [43]:
X_train.to_csv('X_train_prepared.csv', index=False)
y_train.to_csv('y_train_prepared.csv', index=False)

In [44]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [45]:
catboost_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'eval_metric': 'AUC',
    'od_type': 'Iter',
    'od_wait': 100,
    'random_seed': 42,
    'verbose': 100
}

In [47]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(X_train.shape[0])
#test_preds = np.zeros(X_test.shape[0])

In [48]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold + 1}")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = CatBoostClassifier(**catboost_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        cat_features=cat_features,
        use_best_model=True,
        verbose=100
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    #test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold + 1} AUC: {fold_auc:.5f}")


Fold 1
0:	test: 0.5595886	best: 0.5595886 (0)	total: 77.4ms	remaining: 1m 17s
100:	test: 0.7618680	best: 0.7618680 (100)	total: 3.1s	remaining: 27.6s
200:	test: 0.7701236	best: 0.7701236 (200)	total: 6.12s	remaining: 24.3s
300:	test: 0.7743662	best: 0.7743662 (300)	total: 9.06s	remaining: 21s
400:	test: 0.7768748	best: 0.7768928 (399)	total: 12s	remaining: 17.9s
500:	test: 0.7786514	best: 0.7786514 (500)	total: 14.8s	remaining: 14.8s
600:	test: 0.7796226	best: 0.7796226 (600)	total: 17.7s	remaining: 11.8s
700:	test: 0.7803895	best: 0.7803970 (699)	total: 20.5s	remaining: 8.76s
800:	test: 0.7809665	best: 0.7809697 (793)	total: 23.4s	remaining: 5.8s
900:	test: 0.7814782	best: 0.7814798 (899)	total: 26.2s	remaining: 2.88s
999:	test: 0.7819508	best: 0.7819508 (999)	total: 29s	remaining: 0us

bestTest = 0.7819507717
bestIteration = 999

Fold 1 AUC: 0.78195

Fold 2
0:	test: 0.5615141	best: 0.5615141 (0)	total: 27.6ms	remaining: 27.5s
100:	test: 0.7697495	best: 0.7697495 (100)	total: 3.04s	r

In [49]:
total_auc = roc_auc_score(y_train, oof_preds)
print(f"\nTotal OOF AUC: {total_auc:.5f}")


Total OOF AUC: 0.78578
